In [1]:
import json
import numpy as np
import os

In [2]:
path = "/home/jamesdin/James/ThinkingWithVideos/data/MultiTaskVideoReasoning/MTVR_Tool_CoT/vidchapters.json"
data = json.loads(open(path, 'r').read())
data[0]

{'id': 'tool1_000003',
 'tool_params': '[35.79, 105.77]',
 'duration': 189.0,
 'solution': ['<think>Okay, let\'s get this done. I need to pinpoint the exact start and end times for the text "Don\'t bite back" appearing on screen. Right, first things first, I\'ll scan through the video frames.\n\nAha! At `84.28s`, the text pops into view. Now I\'ll scrub forward to see when it disappears. Hmm...it\'s still there at `84.78s`, then at `85.27s`, `85.77s`, and it persists through `86.27s`, `86.77s`, and `87.27s`. The text is still visible at `87.77s`. But... it\'s gone at `88.27s`. Okay, so the event starts around `84.28s` and ends around `88.27s`.\n\nLet\'s refine these timings. I want the exact frame. The text, "DON\'T BITE BACK," first appears, and is *clearly* visible, at `84.28s`. It\'s still there at `87.77s`, and then gone at `88.27s`. The duration is from the frame it *first* appears until the last one where it is visible.\n\nTherefore:\n\nStart time: `84.28s`.\n\nEnd time: Ah, this

In [3]:
len(data)

10571

In [4]:
unique_videos = set()

for d in data:
    unique_videos.add(d['video'])

len(unique_videos)


2737

In [5]:
result_dir = "/data/user_data/jamesdin/outputs/eval/global_step_137/evaluation_maxpix384*384_maxfrm256_number/rextime_val"
# result_dir = "/data/user_data/jamesdin/outputs/eval/Qwen2.5-VL-3B-Instruct/global_step_200/evaluation_maxpix384*384_maxfrm256_number/vsibench"
result_file = "result.json"
result_path = os.path.join(result_dir, result_file)

In [6]:
# assume it's json:
try:
    results = json.load(open(result_path))
except:
    # it's jsonl
    results = [json.loads(line) for line in open(result_path)]
len(results)

763

In [7]:
if isinstance(results, list):
    results = {result['id']: result for result in results}


In [8]:
count = 0  

for key, value in results.items():
    if '<answer>' in value['pred']:
        count += 1

print(count)

698


In [9]:
tool_call_keys = []
count = 0  

for key, value in results.items():
    if 'tool_call' in value['pred']:
        tool_call_keys.append(key)
        count += 1

print(count)




451


In [10]:
key = tool_call_keys[2]

print(results[key]['pred'])







Okay, let's break this down. The user wants me to figure out how a man can comfortably lie down on the floor while talking, based on the video provided. First, I need to understand what the video is showing. It's a man doing some kind of exercise routine on a yoga mat, right? He's lying on his back, doing crunches, and then transitioning into other movements. The key here is to find the part where he's lying down and talking.

Alright, so I'll go through the timestamps. The man starts by standing up, then he sits down on the mat, and then he lies down. The crucial part is when he's lying down. Let's see... at around 117.00 seconds, he's lying on his back, and then he starts talking. He's still on the mat, but he's not sitting up. He's lying down, and he's talking. That seems like the perfect moment to answer the question.

Now, I need to pinpoint the exact time range. The man is lying down and talking from about 117.00 seconds to 121.00 seconds. That's the core of the action. So, the a

In [8]:
results[key]

{'id': '7',
 'question': 'How many stool(s) are in this room?',
 'answer': '2',
 'pred': 'Okay, let\'s see. The user is asking how many stools are in the room. I need to figure out where the stools are located based on the video.\n\nFirst, I\'ll look at the video frames. In the early frames, there\'s a kitchen area with a table and chairs. Then, as the camera moves, it shows a dining area with a large wooden table and several chairs. There\'s also a bench near the table. \n\nWait, the user mentioned "stool(s)", so maybe they\'re referring to the bench. Let me check the frames again. In the later frames, specifically around 97-100 seconds, there\'s a bench with black cushions. It looks like a long bench, possibly used for seating during meals. \n\nI should confirm if this bench is considered a stool. Stools are usually small, portable seats, but benches are longer and often used for seating. However, in some contexts, especially in dining areas, benches might be referred to as stools. G

In [9]:
valid_tool_exec_stats = [results[key]['meta_info']['valid_tool_exec_stats'] for key in tool_call_keys]

In [10]:
np.sum(valid_tool_exec_stats)

np.int64(293)

In [54]:
count = 0
for key in tool_call_keys:
    
    # if results[key]['meta_info']['valid_tool_exec_stats']:
    #     print(results[key]['pred'])
    #     print('-' * 80)
    
    if results[key]['meta_info']['valid_tool_call_stats'] and not results[key]['meta_info']['valid_tool_exec_stats']:
        print(results[key]['pred'])
        print('-' * 80)
    
    count += 1


Okay, let's see. The user wants to know how many tables are in the room. I need to look at the video frames and count the tables.

First, I'll scan through the video. There's a coffee table near the Christmas tree. That's one. Then, there's a dining table with chairs around it. That's another. Wait, is there any other table? Let me check again. The video shows a TV stand, but that's not a table. The bookshelf has some items, but no table. The side table next to the couch is also not a table. So, I think there are two tables: the coffee table and the dining table. 

Wait, maybe I missed something. Let me recheck. The coffee table is definitely there. The dining table is clearly visible. No other tables. So, the answer should be two.
</think>
<tool_call>
{"name": "get_video_clip_frame", "arguments": {"start_time": 13.0, "end_time": 14.0}}
</tool_call>Okay, so the user asked how many tables are in the room. I need to figure out the answer based on the video frames provided.

First, I'll g

In [55]:
count

165

# Check if tool results are appended to context


In [ ]:
def parse_conversation(text):
    """Parse the conversation structure to see if tool results are present."""
    # Split by im_start/im_end tags
    messages = []
    parts = text.split('<|im_start|>')
    
    for part in parts[1:]:  # Skip first empty part
        if '<|im_end|>' in part:
            role_and_content = part.split('<|im_end|>')[0]
            lines = role_and_content.split('\n', 1)
            role = lines[0]
            content = lines[1] if len(lines) > 1 else ''
            messages.append({
                'role': role,
                'content': content
            })
    
    return messages

# Check a sample with tool calls
sample_key = tool_call_keys[2]
sample = results[sample_key]

print(f"Sample ID: {sample_key}")
print(f"\nNumber of tool calls in pred: {sample['pred'].count('<tool_call>')}")
print(f"Number of <tool_response> tags in text: {sample['text'].count('<tool_response>')}")
print("\n" + "="*80)

# Parse the conversation
messages = parse_conversation(sample['text'])
print(f"\nTotal conversation turns: {len(messages)}")

for i, msg in enumerate(messages):
    print(f"\n--- Message {i} ({msg['role']}) ---")
    if 'tool_response' in msg['content'].lower() or 'tool_call' in msg['content'].lower():
        print(msg['content'][:500] + "..." if len(msg['content']) > 500 else msg['content'])
    else:
        print(f"Length: {len(msg['content'])} chars")
        if i > 0:  # Show snippet for non-system messages
            print(msg['content'][:200] + "..." if len(msg['content']) > 200 else msg['content'])


In [ ]:
# Check if tool responses contain visual content
def check_tool_responses(text):
    """Check what's actually in the tool response."""
    results = {
        'has_tool_response': '<tool_response>' in text,
        'num_tool_responses': text.count('<tool_response>'),
        'has_vision_tokens': '<|vision_start|>' in text or '<|video_pad|>' in text,
        'num_vision_blocks': text.count('<|vision_start|>'),
        'tool_response_contents': []
    }
    
    # Extract tool response content
    if results['has_tool_response']:
        parts = text.split('<tool_response>')
        for i, part in enumerate(parts[1:], 1):
            if '</tool_response>' in part:
                response_content = part.split('</tool_response>')[0]
                results['tool_response_contents'].append({
                    'index': i,
                    'length': len(response_content),
                    'has_vision': '<|vision_start|>' in response_content,
                    'has_video_pad': '<|video_pad|>' in response_content,
                    'preview': response_content[:200]
                })
    
    return results

# Check multiple samples
print("Checking tool responses across samples...")
print("="*80)

for i, key in enumerate(tool_call_keys[:5]):  # Check first 5
    print(f"\n### Sample {i}: {key}")
    sample = results[key]
    
    tool_info = check_tool_responses(sample['text'])
    print(f"Has tool response: {tool_info['has_tool_response']}")
    print(f"Number of tool responses: {tool_info['num_tool_responses']}")
    print(f"Number of vision blocks in full text: {tool_info['num_vision_blocks']}")
    
    if tool_info['tool_response_contents']:
        for tr in tool_info['tool_response_contents']:
            print(f"\n  Tool Response {tr['index']}:")
            print(f"    Length: {tr['length']} chars")
            print(f"    Has vision tokens: {tr['has_vision']}")
            print(f"    Has video_pad: {tr['has_video_pad']}")
            print(f"    Preview: {tr['preview']}")
    
    print(f"\nMeta info: valid_tool_call_stats={sample['meta_info']['valid_tool_call_stats']}, "
          f"valid_tool_exec_stats={sample['meta_info']['valid_tool_exec_stats']}")
    print("-"*80)


# Check <think> token generation issue


In [ ]:
# Analyze <think> tag generation patterns
def analyze_think_tags(pred):
    """Analyze where <think> tags appear and their structure."""
    info = {
        'has_think_open': '<think>' in pred,
        'has_think_close': '</think>' in pred,
        'num_think_pairs': pred.count('<think>'),
        'num_tool_calls': pred.count('<tool_call>'),
        'num_answers': pred.count('<answer>'),
        'starts_with_think': pred.strip().startswith('<think>'),
        'positions': []
    }
    
    # Find positions of all tags
    tags = ['<think>', '</think>', '<tool_call>', '</tool_call>', '<answer>', '</answer>']
    for tag in tags:
        pos = 0
        while True:
            pos = pred.find(tag, pos)
            if pos == -1:
                break
            info['positions'].append({'tag': tag, 'pos': pos, 'char': pos})
            pos += len(tag)
    
    # Sort by position
    info['positions'].sort(key=lambda x: x['pos'])
    
    return info

# Check samples with different patterns
print("Analyzing <think> tag generation patterns...")
print("="*80)

samples_with_think = []
samples_without_think = []

for key in tool_call_keys[:50]:
    pred = results[key]['pred']
    if '<think>' in pred:
        samples_with_think.append(key)
    else:
        samples_without_think.append(key)

print(f"Samples with <think>: {len(samples_with_think)}")
print(f"Samples without <think>: {len(samples_without_think)}")
print(f"Percentage with <think>: {len(samples_with_think)/50*100:.1f}%")
print()

# Analyze a few samples with <think>
if samples_with_think:
    print("\n### Sample WITH <think>:")
    key = samples_with_think[0]
    pred = results[key]['pred']
    info = analyze_think_tags(pred)
    
    print(f"ID: {key}")
    print(f"Starts with <think>: {info['starts_with_think']}")
    print(f"Tag sequence:")
    for item in info['positions'][:10]:  # First 10 tags
        print(f"  Pos {item['pos']:6d}: {item['tag']}")
    
    print(f"\nFirst 200 chars of pred:")
    print(pred[:200])

# Analyze samples without <think>
if samples_without_think:
    print("\n" + "="*80)
    print("\n### Sample WITHOUT <think>:")
    key = samples_without_think[0]
    pred = results[key]['pred']
    info = analyze_think_tags(pred)
    
    print(f"ID: {key}")
    print(f"Has <tool_call>: {info['num_tool_calls'] > 0}")
    print(f"Has <answer>: {info['num_answers'] > 0}")
    
    print(f"\nFirst 500 chars of pred:")
    print(pred[:500])


In [ ]:
# Check tokenizer for <think> token
try:
    from transformers import AutoTokenizer
    
    model_path = "/data/user_data/jamesdin/exports/qwen3_vl_2b_thinking_tool_rl_bs8_step1926_hf"
    print(f"Loading tokenizer from: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    # Check if <think> is in vocab
    test_strings = ["<think>", "</think>", "<tool_call>", "<answer>"]
    
    print("\nTokenization test:")
    for s in test_strings:
        tokens = tokenizer.encode(s, add_special_tokens=False)
        decoded = tokenizer.decode(tokens)
        print(f"{repr(s):20s} -> Tokens: {tokens}, Num: {len(tokens)}, Decoded: {repr(decoded)}")
    
    # Check vocab for think-related tokens
    print("\nSearching vocab for 'think' tokens:")
    vocab = tokenizer.get_vocab()
    think_tokens = [(k, v) for k, v in vocab.items() if 'think' in k.lower()]
    for token, idx in sorted(think_tokens, key=lambda x: x[1])[:10]:
        print(f"  ID {idx:6d}: {repr(token)}")
    
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Check if prompt format affects <think> generation
def extract_last_assistant_prefix(text):
    """Extract what comes right before the model's generation."""
    # Find the last assistant turn start
    last_assistant = text.rfind('<|im_start|>assistant')
    if last_assistant == -1:
        return None
    
    # Find the end of that line and get next few chars
    start_pos = text.find('\n', last_assistant)
    if start_pos == -1:
        return None
    
    # Get up to 200 chars after <|im_start|>assistant
    return text[last_assistant:start_pos+200]

print("Checking prompt format before generation...")
print("="*80)

# Compare samples with and without <think>
if samples_with_think and samples_without_think:
    print("\n### Sample WITH <think>:")
    key = samples_with_think[0]
    prefix = extract_last_assistant_prefix(results[key]['text'])
    print(f"ID: {key}")
    print(f"Last assistant prefix:")
    print(prefix if prefix else "Could not extract")
    print(f"\nPrediction starts with:")
    print(results[key]['pred'][:300])
    
    print("\n" + "="*80)
    print("\n### Sample WITHOUT <think>:")
    key = samples_without_think[0]
    prefix = extract_last_assistant_prefix(results[key]['text'])
    print(f"ID: {key}")
    print(f"Last assistant prefix:")
    print(prefix if prefix else "Could not extract")
    print(f"\nPrediction starts with:")
    print(results[key]['pred'][:300])
    
    # Check if the prompt explicitly asks for <think>
    print("\n" + "="*80)
    print("\nChecking if prompt asks for <think> tags:")
    for i, key in enumerate([samples_with_think[0], samples_without_think[0]]):
        sample = results[key]
        has_think_instruction = '<think>' in sample['text'] and '</think>' in sample['text']
        has_think_in_system = False
        
        # Check system message
        if '<|im_start|>system' in sample['text']:
            system_end = sample['text'].find('<|im_end|>', sample['text'].find('<|im_start|>system'))
            system_msg = sample['text'][sample['text'].find('<|im_start|>system'):system_end]
            has_think_in_system = '<think>' in system_msg
        
        label = "WITH <think>" if i == 0 else "WITHOUT <think>"
        print(f"\n{label} (ID: {key}):")
        print(f"  Has <think> in prompt: {has_think_instruction}")
        print(f"  Has <think> in system message: {has_think_in_system}")


## Summary and Diagnosis

Run the cells above to diagnose:

1. **Tool Result Context Issue**: 
   - Check if `<tool_response>` tags exist in the `text` field
   - Check if tool responses contain visual content (`<|vision_start|>`, `<|video_pad|>`)
   - Compare `valid_tool_call_stats` vs `valid_tool_exec_stats` to see if tools are called but not executed

2. **<think> Token Generation Issue**:
   - Check what percentage of samples have `<think>` tags
   - Check if `<think>` is properly tokenized (single token vs multi-token)
   - Check if prompt format includes `<think>` examples
   - Check if the model starts generation with `<think>` or generates it mid-stream

**Common Issues:**
- If tool responses are missing: Tool execution wrapper may not be appending results to context
- If `<think>` is multi-token: Need to add `<think>` and `</think>` as special tokens to tokenizer
- If `<think>` appears mid-generation instead of at start: May need prefix constraint or stronger reward for correct positioning


In [ ]:
# Generate diagnostic report
def generate_diagnostic_report():
    """Generate a comprehensive diagnostic report."""
    print("="*80)
    print("DIAGNOSTIC REPORT")
    print("="*80)
    
    # 1. Overall statistics
    print("\n### 1. OVERALL STATISTICS")
    print(f"Total results: {len(results)}")
    print(f"Samples with tool calls: {len(tool_call_keys)}")
    print(f"Samples with <answer>: {sum(1 for r in results.values() if '<answer>' in r['pred'])}")
    
    # 2. Tool execution stats
    print("\n### 2. TOOL EXECUTION")
    valid_calls = sum(1 for k in tool_call_keys if results[k]['meta_info'].get('valid_tool_call_stats', 0) > 0)
    valid_execs = sum(1 for k in tool_call_keys if results[k]['meta_info'].get('valid_tool_exec_stats', 0) > 0)
    print(f"Samples with valid tool calls: {valid_calls}/{len(tool_call_keys)}")
    print(f"Samples with valid tool executions: {valid_execs}/{len(tool_call_keys)}")
    print(f"Tool execution rate: {valid_execs/valid_calls*100:.1f}%" if valid_calls > 0 else "N/A")
    
    if valid_calls > valid_execs:
        print(f"\n⚠️  WARNING: {valid_calls - valid_execs} samples have tool calls but no executions!")
        print("   This suggests tool results may not be appended to context.")
    
    # 3. Think tag analysis
    print("\n### 3. <THINK> TAG GENERATION")
    samples_with_think = [k for k in tool_call_keys if '<think>' in results[k]['pred']]
    samples_start_with_think = [k for k in samples_with_think if results[k]['pred'].strip().startswith('<think>')]
    
    print(f"Samples with <think> tag: {len(samples_with_think)}/{len(tool_call_keys)} ({len(samples_with_think)/len(tool_call_keys)*100:.1f}%)")
    print(f"Samples starting with <think>: {len(samples_start_with_think)}/{len(samples_with_think)} ({len(samples_start_with_think)/len(samples_with_think)*100:.1f}%)" if samples_with_think else "N/A")
    
    if len(samples_with_think) < len(tool_call_keys) * 0.5:
        print(f"\n⚠️  WARNING: Less than 50% of samples have <think> tags!")
        print("   Possible causes:")
        print("   - <think> not added as special token in tokenizer")
        print("   - Model not properly fine-tuned on <think> format")
        print("   - Prompt format doesn't encourage <think> generation")
    
    # 4. Tool response content analysis
    print("\n### 4. TOOL RESPONSE CONTENT")
    samples_with_response = sum(1 for k in tool_call_keys if '<tool_response>' in results[k].get('text', ''))
    samples_with_vision = sum(1 for k in tool_call_keys if '<|vision_start|>' in results[k].get('text', ''))
    
    print(f"Samples with <tool_response> in text: {samples_with_response}/{len(tool_call_keys)}")
    print(f"Samples with vision tokens in text: {samples_with_vision}/{len(tool_call_keys)}")
    
    if samples_with_response == 0:
        print(f"\n⚠️  CRITICAL: No samples have <tool_response> tags!")
        print("   Tool results are NOT being appended to context.")
        print("   The model cannot use tool outputs because it never sees them.")
    elif samples_with_vision == 0:
        print(f"\n⚠️  WARNING: Tool responses exist but contain no vision tokens!")
        print("   Tool may be returning text-only responses without visual content.")
    
    print("\n" + "="*80)
    print("END OF REPORT")
    print("="*80)

generate_diagnostic_report()


In [ ]:
# Check tokenizer for <think> token
# This requires loading the tokenizer - adjust path as needed
try:
    from transformers import AutoTokenizer
    
    # Replace with your model path
    model_path = "/data/user_data/jamesdin/exports/qwen3_vl_2b_thinking_tool_rl_bs8_step1926_hf"
    
    print(f"Loading tokenizer from: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    # Check if <think> is in vocab
    test_strings = [
        "<think>",
        "</think>",
        "<tool_call>",
        "</tool_call>",
        "<answer>",
        "</answer>",
        "think",
        "< think >",
        "<|think|>",
    ]
    
    print("\nTokenization test:")
    print("="*80)
    for s in test_strings:
        tokens = tokenizer.encode(s, add_special_tokens=False)
        decoded = tokenizer.decode(tokens)
        print(f"\nString: {repr(s)}")
        print(f"  Tokens: {tokens}")
        print(f"  Decoded: {repr(decoded)}")
        print(f"  Num tokens: {len(tokens)}")
        
        # Check if it's a single token
        if len(tokens) == 1:
            token_str = tokenizer.convert_ids_to_tokens(tokens[0])
            print(f"  Token string: {repr(token_str)}")
    
    # Check vocab for think-related tokens
    print("\n" + "="*80)
    print("\nSearching vocab for 'think' related tokens:")
    vocab = tokenizer.get_vocab()
    think_tokens = [(k, v) for k, v in vocab.items() if 'think' in k.lower()]
    think_tokens.sort(key=lambda x: x[1])
    
    for token, idx in think_tokens[:20]:  # Show first 20
        print(f"  Token ID {idx:6d}: {repr(token)}")
    
    # Check special tokens
    print("\n" + "="*80)
    print("\nSpecial tokens:")
    print(f"  bos_token: {repr(tokenizer.bos_token)}")
    print(f"  eos_token: {repr(tokenizer.eos_token)}")
    print(f"  pad_token: {repr(tokenizer.pad_token)}")
    print(f"  unk_token: {repr(tokenizer.unk_token)}")
    
    if hasattr(tokenizer, 'additional_special_tokens'):
        print(f"\nAdditional special tokens ({len(tokenizer.additional_special_tokens)}):")
        for tok in tokenizer.additional_special_tokens[:20]:
            print(f"    {repr(tok)}")
    
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("\nPlease update the model_path variable to point to your model.")
